In [5]:
import pandas as pd
import os
import tempfile
from pathlib import Path

dor_district_map = {
    "440":"Greater LA District",
    "Greater La District":"Greater LA District",
    "La South Bay District":"LA South Bay District"
}

# User home directory (dynamic)
home_dir = Path.home()

# SharePoint relative path inside OneDrive
sharepoint_relative_input_path = Path(
    "APM US",
    "Data and Insights - Documents",
    "Power BI Source Files",
    "DOR-AJCC Collab",
    "Files for Python"
)

base_dir = home_dir / sharepoint_relative_input_path
# 🔥 File for conversion
file_substring = "DOR Staff Survey"

# 🧐 Get files matching substring
matching_files = [
    f for f in base_dir.glob("*.xlsx")
    if file_substring in f.name
]
# ☠️ No files matching error message
if not matching_files:
    raise FileNotFoundError("No matching Excel file found.")
    
# ☠️ Muiltiple files matching substring error message
if len(matching_files) > 1:
    raise ValueError(
        f"Multiple matching files found:\n{[f.name for f in matching_files]}"
    )

input_path = matching_files[0]

print(f"✅ Using file: {input_path}")

# ✔️ Declare path to write new file to
sharepoint_relative_output_path = Path(
    "APM US",
    "Data and Insights - Documents",
    "Power BI Source Files",
    "DOR-AJCC Collab"
)

output_base_dir = home_dir / sharepoint_relative_output_path

# ✔️ Full path with name for output file 
output_path = rf'{output_base_dir}\{file_substring}.xlsx'
# output_path = rf'C:\Users\StevenFoster\Downloads\{file_name}.xlsx'

# 🛻 Load the 1st worksheet with NO headers
df_raw = pd.read_excel(input_path, sheet_name=0, header=None)

# 🛻 Find the row index where column 0 == "DOR_District"
header_row_idx = df_raw.index[df_raw.iloc[:, 1] == "DOR_District"]

# ☠️ Error message when column header not located
if header_row_idx.empty:
    raise ValueError("Header row containing 'DOR_District' not found.")

header_row_idx = header_row_idx[0]

# ➖ Remove rows above the header
df_clean = df_raw.iloc[header_row_idx:].reset_index(drop=True)

# 🛻 Promote that row to headers
df_clean.columns = df_clean.iloc[0]
df_clean = df_clean.iloc[1:].reset_index(drop=True)

# ✔️ Clean column names
df_clean.columns = (
    df_clean.columns
        .astype(str)
        .str.replace('\n', ' ', regex=False)  # replace newline with space
        .str.replace(r'\s+', ' ', regex=True) # collapse multiple spaces
        .str.strip()                           # trim leading/trailing spaces
)

# ➖ Remove repeated header rows inside the data
df_clean = df_clean[df_clean["DOR_District"] != "DOR_District"]

# ➖ Remove total row
df_clean = df_clean[~df_clean["DOR_District"].astype(str).str.contains(
    "Total", na=False
)]

# 4️⃣ Validate columns are present 
expected_string_cols = [
                        "DOR_District",
                        "Modified By",
                        "Other_Districts",
                        "Role",
                        "Other_Role",
                        "Admin_Connection_to_WDB",
                        "Admin_System_Alignment",
                        "Admin_Barriers_1",
                        "Admin_Role_Clarity",
                        "Admin_Barriers_2",
                        "Admin_Opportunities",
                        "Interaction_Frequency",
                        "Admin_CoLocation_Effectiveness",
                        "CoLocation_Presence",
                        "CoLocation_Effectiveness",
                        "AJCC_Value",
                        "AJCC_Staff_Understanding",
                        "AJCC_Accessibility",
                        "AJCC_Service_Quality",
                        "Partnership_Strength",
                        "Communication_Effectiveness",
                        "Role_Clarity",
                        "Collaboration_Structure",
                        "Referral_Knowledge",
                        "Referral_Process_Clarity",
                        "Referral_Confidence",
                        "Referral_Timeliness",
                        "Service_Coordination",
                        "Comfort_Working_With_AJCC",
                        "AJCC_Responsiveness",
                        "Mutual_Respect",
                        "Training_Sufficiency",
                        "Training_Need",
                        "Training_Priority_Areas",
                        "Collaboration_Challenges",
                        "Collaboration_Strengths",
                        "Collaboration_Improvements",
                        "Overall_Collaboration_Level",
                        "Additional_Feedback",
                        "Other_A1",
                        "Other_Collaboration_Challenges",
                        "Other_Collaboration_Strengths",
                        "Other_Collaboration_Improvements"
                        ]


for col in expected_string_cols:
    if col not in df_clean.columns:
        raise ValueError(f"❌ Missing expected string column: '{col}'")

    # Convert to string
    df_clean[col] = df_clean[col].astype("string")

    # Verify all values are strings (or NA)
    bad_mask = df_clean[col].apply(lambda v: not (pd.isna(v) or isinstance(v, str)))
    if bad_mask.any():
        bad_values = df_clean.loc[bad_mask, col].head(10)
        raise TypeError(
            f"❌ Column '{col}' contains non-string data.\n"
            f"Example invalid values:\n{bad_values}"
        )

print("✅ All column types validated successfully. Safe to continue.")

# 🤖 Capitlize headers
df_clean.columns = df_clean.columns.str.upper()

# 🤖 Drop blank rows, fail back for "used range" in excel file
df_clean = df_clean.dropna(how="all")

# 🤖 Trim all string columns 
df_clean = df_clean.apply(
    lambda col: col.str.strip() if col.dtype == "string" else col
)

# ➕ Map REGION CODE
df_clean["DOR_DISTRICT"] = df_clean["DOR_DISTRICT"].replace(dor_district_map)

df = df_clean.copy()

# 🤖 Convert output path to pathlib path 
output_path = Path(output_path)

# 🤖 Check if file exists and remove 
if output_path.exists():
    print(f"⚠️ Overwriting existing file: {output_path}")
    output_path.unlink()  # deletes the file
else: 
    print(f"🆕 File does not exist. Creating new file at: {output_path}")
    
# ✍️ Export to Excel
df.to_excel(output_path, index=False)

✅ Using file: C:\Users\StevenFoster\APM US\Data and Insights - Documents\Power BI Source Files\DOR-AJCC Collab\Files for Python\DOR Staff Survey 6.5.26.xlsx
✅ All column types validated successfully. Safe to continue.
⚠️ Overwriting existing file: C:\Users\StevenFoster\APM US\Data and Insights - Documents\Power BI Source Files\DOR-AJCC Collab\DOR Staff Survey.xlsx
